## Exempel ML-modellering Titanic

Titanic-datasetet bygger på passagerare ombord på RMS Titanic år 1912 och innehåller information om ålder, kön, biljettklass, familjeförhållanden och om passageraren överlevde eller inte.

Vårt mål är att, baserat på denna information, bygga en modell som kan förutsäga sannolikheten att en passagerare överlever.

Problemtyp: Klassificering

### Ladda in bibliotek

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score

### Ladda in dataset

In [ ]:
df_titanic = pd.read_csv("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")

In [ ]:
list(df_titanic.columns)

'PassengerId' = Unikt ID. 
'Survived' = Målvariabel. 0 = dog, 1 = överlevde
'Pclass' = Biljettklass. 1, 2 eller 3. 
'Name'
'Sex'
'Age'
'SibSp' = Antal syskon + make/maka ombord
'Parch' = Antal föräldrar + barn ombord
'Ticket' = Biljettnummer.
'Fare' = Biljettpris. 
'Cabin' = Hyttrum.
'Embarked' = Ombordstigningshamn. C = Cherbourg, Q = Queenstown, S = Southhampton

In [ ]:
# Dela upp dataset i X (features) och y (målvariabel), sedan i träning och test
X = df_titanic.drop("Survived", axis=1)
y = df_titanic["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Dela upp direkt för att undvika dataläckage! 

# Skapa en EDA-dataframe 
train_df = X_train.copy()
train_df["Survived"] = y_train
# Nu har vi en df med alla features och målvariabeln, men endast träningsdata
# Perfekt för EDA! 

### EDA

In [ ]:
train_df.info()

In [ ]:
train_df.head()

Snabb observation:

* Saknade värden i Age och Cabin.
* Kategoriska variabler: Sex, Class, Embarked

### Modellering

Ingen EDA, ingen bearbetning av data - En första naiv modell

In [ ]:
features = ["Age", "Fare"]
X_train_1 = X_train[features]

X_train_1 = X_train_1.fillna(X_train_1.mean())  # enkel imputering
print(X_train_1.info())


model = LogisticRegression()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv_scores = cross_val_score(
    model,
    X_train_1,
    y_train,
    cv=cv,
    scoring="accuracy"
)

accuracy_1 = cv_scores.mean()
print(accuracy_1)

### EDA 

Enkel EDA med tydliga insikter

In [ ]:
train_df.groupby("Sex")["Survived"].mean()

In [ ]:
train_df.groupby("Pclass")["Survived"].mean()

Från EDA vet vi nu att Kön och Klass är extremt viktigt. Lägger till kategoriska variabler. 

### Modellering

In [ ]:
num_features = ["Age", "Fare"]
cat_features = ["Pclass", "Sex"]
X_train_2 = X_train[num_features + cat_features]

X_train_2[num_features] = X_train_2[num_features].fillna(X_train_2[num_features].mean())  # enkel imputering
ohe = OneHotEncoder(
    drop="first",
    sparse_output=False,
    handle_unknown="ignore"
)
X_train_2_cat = ohe.fit_transform(X_train_2[cat_features])
X_train_2_final = np.hstack([X_train_2[num_features], X_train_2_cat])

model = LogisticRegression()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
cv_scores = cross_val_score(
    model,
    X_train_2_final,
    y_train,
    cv=cv,
    scoring="accuracy"
)

accuracy_2 = cv_scores.mean()
print(accuracy_2)

Modellen blev mycket bättre efter att vi inkluderade Klass och Kön.

### EDA

In [ ]:
train_df.groupby(pd.cut(train_df["Age"], bins=[0,12,18,60,100]))["Survived"].mean()

Barn överlever oftare än vuxna. Sambandet är inte linjärt. Nu har vi två alternativ: 

* Strategi 1 – Gör datan mer linjär (feature engineering). Vi kodar den domänkunskap vi hittat direkt i datan.
* Strategi 2 – Använd en modell som klarar olinjärhet. Vi låter modellen själv upptäcka mönstret.

Vi testar! 

### Modellering

In [ ]:
X_train["is_child"] = (X_train["Age"] < 12).astype(int)

num_features = ["Fare"]
cat_features = ["Sex", "Pclass"]
binary_features = ["is_child"]

X_train_3 = X_train[num_features + cat_features + binary_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
        ("bin", "passthrough", binary_features)
    ]
)

model = LogisticRegression()

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train_3,
    y_train,
    cv=cv,
    scoring="accuracy"
)


accuracy_3 = cv_scores.mean()
print(accuracy_3)

In [ ]:
print("Modell 1 (endast numeriskt):", accuracy_1)
print("Modell 2 (+ kategoriskt):", accuracy_2)
print("Modell 3 (+ feature engineering):", accuracy_3)

In [ ]:
num_features = ["Age", "Fare"]
cat_features = ["Sex", "Pclass"]

X_train_4 = X_train[num_features + cat_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train_4,
    y_train,
    cv=cv,
    scoring="accuracy"
)


accuracy_4 = cv_scores.mean()
print(accuracy_4)

In [ ]:
print("Modell 1 (endast numeriskt):", accuracy_1)
print("Modell 2 (+ kategoriskt):", accuracy_2)
print("Modell 3 (+ feature engineering):", accuracy_3)
print("Modell 4 (Random Forest):", accuracy_4)

### EDA

Inkludera inte PassengerId, Name och Ticket. Unika för varje passagerare - säger oss ingenting. Leder till overfitting om man kategoriserar. 

Cabin har många saknade värden. Första bokstaven kan indikera socioekonomisk status, kan vara värt att göra feature engineering och endast ta med bokstaven. Finns många olika bokstäver. Vi tar inkluderar inte Cabin. 

Embarked är en kategorisk variabel med få kategorier. Kan påverka indirekt. Resväg kan säga något om klass. Vi inkluderar Embarked. 

In [ ]:
import matplotlib.pyplot as plt

num_cols = ["Age", "Fare", "SibSp", "Parch", "Survived"]
corr = train_df[num_cols].corr()

fig, ax = plt.subplots(figsize=(8, 6))

cax = ax.imshow(corr.values)
plt.colorbar(cax)
ax.set_xticks(np.arange(len(corr.columns)))
ax.set_yticks(np.arange(len(corr.columns)))
ax.set_xticklabels(corr.columns)
ax.set_yticklabels(corr.columns)
plt.xticks(rotation=45, ha="right")

for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(
            j, i,
            f"{corr.iloc[i, j]:.2f}",
            ha="center", va="center", color="black"
        )

ax.set_title("Korrelationsmatris (numeriska variabler)")
plt.tight_layout()

plt.show()

Notera att den visar endast linjära samband! 

Vi vet att ålder har en stark effekt, men låg korrelation i matris. 

In [ ]:
train_df["family_size"] = train_df["SibSp"] + train_df["Parch"] + 1

In [ ]:
train_df.groupby("family_size")["Survived"].mean()

In [ ]:
train_df.groupby("family_size")["Survived"].count()

In [ ]:
counts = train_df["family_size"].value_counts().sort_index()

plt.figure(figsize=(6, 4))
plt.bar(counts.index, counts.values)
plt.xlabel("Family size")
plt.ylabel("Number of passengers")
plt.title("Distribution of family size")
plt.show()

2–4 personer i familjen verkar ha högst överlevnad

Ensam eller väldigt stor familj har en lägre överlevnad

Få observationer för stora familjer. Hög varians. --> Osäkra slutsatser. 

En överlevnadsandel på 0 eller 1 kan bero på slump. 

Motiverar gruppering --> Slå ihop kategorier. 

Skev fördelning betyder inte att något är oviktigt – det betyder att vi måste vara försiktiga.

### Modellering

In [ ]:
X_train["is_child"] = (X_train["Age"] < 12).astype(int)

X_train["family_size"] = X_train["SibSp"] + X_train["Parch"] + 1
X_train["is_alone"] = (X_train["family_size"] == 1).astype(int)
X_train["small_family"] = X_train["family_size"].between(2, 4).astype(int)

num_features = ["Fare"]
cat_features = ["Sex", "Pclass", "Embarked"]
binary_features = ["is_child", "is_alone", "small_family"]

X_train_5 = X_train[num_features + cat_features + binary_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
        ("bin", "passthrough", binary_features)
    ]
)

model = LogisticRegression()

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cv_scores = cross_val_score(
    pipeline,
    X_train_5,
    y_train,
    cv=cv,
    scoring="accuracy"
)


accuracy_5 = cv_scores.mean()
print(accuracy_5)

Låt oss se om vi kan förbättra vår RandomForest-modell med optimering av hyperparametrar.

In [ ]:
num_features = ["Age", "Fare", "family_size"]
cat_features = ["Sex", "Pclass", "Embarked"]

X_train_6 = X_train[num_features + cat_features]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="mean"))
])

categorical_transformer = Pipeline([
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features)
    ]
)

model = RandomForestClassifier()

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_leaf": [1, 5, 10],
    "model__max_features": ["sqrt", 0.5]
}

pipeline = Pipeline([
    ("preprocess", preprocessor),
    ("model", model)
])

grid = GridSearchCV(
    pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=-1
)

grid.fit(X_train_6, y_train)
accuracy_6 = grid.best_score_

print("Accuracy:", accuracy_6)
print("Bästa hyperparametrar:")
print(grid.best_params_)

In [ ]:
print("Modell 1 (endast numeriskt):", accuracy_1)
print("Modell 2 (+ kategoriskt):", accuracy_2)
print("Modell 3 (+ feature engineering):", accuracy_3)
print("Modell 4 (Random Forest):", accuracy_4)
print("Modell 5 (+ mer feature engineering och feature selection):", accuracy_5)
print("Modell 6 (Random Forest + hyperparameter tuning):", accuracy_6)

“ML är inte att välja rätt modell – det är att ställa rätt frågor till datan.”

EDA → hypotes → bearbetning → ny modell → utvärdering

### Utvärdering

Dags att använda test-setet! 

In [185]:
X_test = X_test.copy()
X_test["family_size"] = X_test["SibSp"] + X_test["Parch"] + 1
num_features = ["Age", "Fare", "family_size"]
cat_features = ["Sex", "Pclass", "Embarked"]

X_test = X_test[num_features + cat_features]

In [186]:
final_model = grid.best_estimator_

y_test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, y_test_pred)
print("Test accuracy:", test_accuracy)

Test accuracy: 0.8044692737430168


In [ ]:
type(final_model)

### Lägg till ny passagerare

Kommer den att överleva? 

In [ ]:
new_passenger = pd.DataFrame({
    "Age": [25],
    "Fare": [80],
    "Sex": ["male"],
    "family_size": [3],
    "Pclass": [3],
    "Embarked": ["C"]
})

prediction = final_model.predict(new_passenger)

if prediction[0] == 1:
    print("Passageraren förväntas överleva")
else:
    print("Passageraren förväntas inte överleva")

proba = final_model.predict_proba(new_passenger)
print(f"Sannolikhet att överleva: {proba[0][1]:.2%}")